# Generate Source Files - Netflix Titles Split

Splits the existing Netflix titles CSV (from Lab 2) into ~1000 smaller files, to
give Auto Loader a realistic incremental-ingestion source with many files instead
of one large file.

## 1. Configuration

In [0]:
%run ./lab3_00_config

In [0]:
source_path = f"{storage_root}/ingestion/netflix/"                 # existing files from Lab 2
stream_source_path = f"{storage_root}/ingestion/netflix_stream/"   # new folder for this lab, ~1000 files

target_file_count = 1000

print("Reading from:", source_path)
print("Writing split files to:", stream_source_path)

## 2. Read the existing single-file dataset

In [0]:
from pyspark.sql import functions as F

df_all = spark.read.option("header", "true").csv(source_path)
total_rows = df_all.count()

print("Total rows read:", total_rows)

## 3. Repartition and write as ~1000 files

In [0]:
# Repartition into ~1000 files and write each as a separate CSV with header
df_repartitioned = df_all.repartition(target_file_count)

(
    df_repartitioned.write
    .mode("overwrite")
    .option("header", "true")
    .csv(stream_source_path)
)

print(f"Written to {stream_source_path}")

## 4. Verify the file count

In [0]:
files = dbutils.fs.ls(stream_source_path)
csv_files = [f for f in files if f.name.endswith(".csv")]

print("Total files written:", len(csv_files))
print("Example file names:")
for f in csv_files[:5]:
    print(" -", f.name)